In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Load training and testing data
train_df = pd.read_csv('../data/train.csv')
test_df = pd.read_csv('../data/test.csv')

In [7]:
# Minimal preprocessing: drop unnecessary columns, fill NaN in 'Arrival Delay in Minutes' with 0, encode categorical variables
import time
from sklearn.metrics import roc_auc_score

train_df_clean = train_df.drop(['Unnamed: 0', 'id'], axis=1)
test_df_clean = test_df.drop(['Unnamed: 0', 'id'], axis=1)

train_df_clean['Arrival Delay in Minutes'] = train_df_clean['Arrival Delay in Minutes'].fillna(0)
test_df_clean['Arrival Delay in Minutes'] = test_df_clean['Arrival Delay in Minutes'].fillna(0)

# Encode categorical features and target
categorical_cols = ['Gender', 'Customer Type', 'Type of Travel', 'Class']
train_df_encoded = pd.get_dummies(train_df_clean, columns=categorical_cols, drop_first=True)
test_df_encoded = pd.get_dummies(test_df_clean, columns=categorical_cols, drop_first=True)

# Ensure test_df has the same columns as train_df
test_df_encoded = test_df_encoded.reindex(columns=train_df_encoded.columns, fill_value=0)

# Separate features and target
X_train = train_df_encoded.drop('satisfaction', axis=1)
# Convert string targets to binary for ROC AUC calculation
y_train = (train_df_encoded['satisfaction'] == 'satisfied').astype(int)

X_test = test_df_encoded.drop('satisfaction', axis=1)
y_test = (test_df_encoded['satisfaction'] == 'satisfied').astype(int)

# Train baseline LogisticRegression model
model = LogisticRegression(max_iter=1000)

# Measure training runtime
start_time = time.time()
model.fit(X_train, y_train)
end_time = time.time()

training_time = end_time - start_time

# Predict and evaluate using ROC AUC
y_pred_proba = model.predict_proba(X_test)[:, 1]
roc_auc = roc_auc_score(y_test, y_pred_proba)

print(f'Training Runtime: {training_time:.4f} seconds')
print(f'Baseline Model ROC AUC: {roc_auc:.4f}')

Training Runtime: 3.2949 seconds
Baseline Model ROC AUC: 0.9197


c:\Users\jonah\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
